# Recurrent Neural Networks
You should build an end-to-end machine learning pipeline using a recurrent neural network model. In particular, you should do the following:
- Load the `jena climate` dataset using [Pandas](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html). You can find this dataset in the [keras repository](https://keras.io/examples/timeseries/timeseries_weather_forecasting/).
- Split the dataset into training, validation, and test sets. Note that you cannot split time series using [Scikit-Learn](https://keras.io/examples/timeseries/timeseries_weather_forecasting/).
- Build an end-to-end machine learning pipeline, including a [recurrent neural network](https://keras.io/examples/timeseries/timeseries_weather_forecasting/) model.
- Optimize your pipeline by validating your design decisions.
- Test the best pipeline on the test set and report various [evaluation metrics](https://scikit-learn.org/0.15/modules/model_evaluation.html).  
- Check the documentation to identify the most important hyperparameters, attributes, and methods of the model. Use them in practice.

1.load the data

2.Sample the dataset by choosing one measurement per

3.Create target label by shifting the temperaturecoloumn one row/day up and add it as taeget coloumn to your dataset

4.Split the data into train,validation,and test using timestamps.do not shuffle the data or break the time elements

5.Create sequence matrixes using the function in the tutorial

6.Build and train the model

7.Experiment with modal to maximize validation score

8.Test the best model on the test set




#Import the data

In [4]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras

url = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip"
df = pd.read_csv(url, compression="zip")

print(df.shape)
df.head()

(420551, 15)


,Date Time,p (mbar),T (degC),Tpot (K),Tdew (degC),rh (%),VPmax (mbar),VPact (mbar),VPdef (mbar),sh (g/kg),H2OC (mmol/mol),rho (g/m**3),wv (m/s),max. wv (m/s),wd (deg)
0,01.01.2009 00:10:00,996.52,-8.02,265.40,-8.90,93.3,3.33,3.11,0.22,1.94,3.12,1307.75,1.03,1.75,152.3
1,01.01.2009 00:20:00,996.57,-8.41,265.01,-9.28,93.4,3.23,3.02,0.21,1.89,3.03,1309.80,0.72,1.50,136.1
2,01.01.2009 00:30:00,996.53,-8.51,264.91,-9.31,93.9,3.21,3.01,0.20,1.88,3.02,1310.24,0.19,0.63,171.6
3,01.01.2009 00:40:00,996.51,-8.31,265.12,-9.07,94.2,3.26,3.07,0.19,1.92,3.08,1309.19,0.34,0.50,198.0
4,01.01.2009 00:50:00,996.51,-8.27,265.15,-9.04,94.1,3.27,3.08,0.19,1.92,3.09,1309.00,0.32,0.63,214.3


#Sample the data

In [5]:
df["Date Time"] = pd.to_datetime(df["Date Time"], format="mixed", dayfirst=True)
# set datetime index
df = df.set_index("Date Time")

# daily sampling (mean per day)
df_daily = df.resample("D").mean()

print(df_daily.shape)
df_daily.head()

(2923, 14)


,p (mbar),T (degC),Tpot (K),Tdew (degC),rh (%),VPmax (mbar),VPact (mbar),VPdef (mbar),sh (g/kg),H2OC (mmol/mol),rho (g/m**3),wv (m/s),max. wv (m/s),wd (deg)
Date Time,,,,,,,,,,,,,,
2009-01-01,999.145594,-6.810629,266.414545,-8.015594,91.086014,3.691119,3.355524,0.335315,2.091049,3.357832,1305.178252,0.778601,1.378252,181.863077
2009-01-02,999.600625,-3.728194,269.463194,-4.824861,92.086806,4.640069,4.267292,0.373056,2.659792,4.268750,1290.353194,1.419514,2.227361,125.072014
2009-01-03,998.548611,-5.271736,268.002292,-9.015833,76.458056,4.184792,3.107708,1.077014,1.937778,3.111944,1297.117014,1.250903,2.065069,190.383333
2009-01-04,988.510694,-1.375208,272.685347,-2.897014,89.417361,5.524306,4.938958,0.584861,3.114028,4.997014,1264.634514,1.720417,3.564861,213.069861
2009-01-05,990.405694,-4.867153,269.039306,-6.797292,86.260417,4.362708,3.806736,0.555625,2.397014,3.847778,1284.372778,3.800278,5.940000,118.287361


**Create Target (shift temperature by 1 day)**

In [6]:
df_daily["target"] = df_daily["T (degC)"].shift(-1)

# remove last row (NaN target)
df_daily = df_daily.dropna()

df_daily.head()

,p (mbar),T (degC),Tpot (K),Tdew (degC),rh (%),VPmax (mbar),VPact (mbar),VPdef (mbar),sh (g/kg),H2OC (mmol/mol),rho (g/m**3),wv (m/s),max. wv (m/s),wd (deg),target
Date Time,,,,,,,,,,,,,,,
2009-01-01,999.145594,-6.810629,266.414545,-8.015594,91.086014,3.691119,3.355524,0.335315,2.091049,3.357832,1305.178252,0.778601,1.378252,181.863077,-3.728194
2009-01-02,999.600625,-3.728194,269.463194,-4.824861,92.086806,4.640069,4.267292,0.373056,2.659792,4.268750,1290.353194,1.419514,2.227361,125.072014,-5.271736
2009-01-03,998.548611,-5.271736,268.002292,-9.015833,76.458056,4.184792,3.107708,1.077014,1.937778,3.111944,1297.117014,1.250903,2.065069,190.383333,-1.375208
2009-01-04,988.510694,-1.375208,272.685347,-2.897014,89.417361,5.524306,4.938958,0.584861,3.114028,4.997014,1264.634514,1.720417,3.564861,213.069861,-4.867153
2009-01-05,990.405694,-4.867153,269.039306,-6.797292,86.260417,4.362708,3.806736,0.555625,2.397014,3.847778,1284.372778,3.800278,5.940000,118.287361,-15.482847


**Train / Validation / Test Split (NO shuffle)**

In [7]:
n = len(df_daily)

train_end = int(n * 0.7)
val_end   = int(n * 0.85)

train_df = df_daily.iloc[:train_end]
val_df   = df_daily.iloc[train_end:val_end]
test_df  = df_daily.iloc[val_end:]

print(train_df.shape, val_df.shape, test_df.shape)

(2043, 15) (438, 15) (438, 15)


#Normalize

In [8]:
train_mean = train_df.mean()
train_std = train_df.std()

train_df = (train_df - train_mean) / train_std
val_df   = (val_df - train_mean) / train_std
test_df  = (test_df - train_mean) / train_std

#Create sequence Matrixes

In [9]:
sequence_length = 4   # use past 7 days to predict next day
batch_size = 16
def create_dataset(data):
    return keras.utils.timeseries_dataset_from_array(
        data=data.iloc[:-sequence_length].values,
        targets=data["target"].iloc[sequence_length:].values,
        sequence_length=sequence_length,
        batch_size=batch_size
    )
train_ds = create_dataset(train_df)
val_ds   = create_dataset(val_df)
test_ds  = create_dataset(test_df)

#Train Dataset

In [29]:
# model = keras.Sequential([
#     keras.layers.Input(shape=(sequence_length, train_df.shape[1])),

#     keras.layers.LSTM(64),
#     keras.layers.Dense(32, activation="relu"),

#     keras.layers.Dense(1)
# ])


#Model 1 (Deeper LSTM)

# model = keras.Sequential([
#     keras.layers.Input(shape=(sequence_length, train_df.shape[1])),

#     keras.layers.LSTM(128, return_sequences=True),
#     keras.layers.Dropout(0.2),

#     keras.layers.LSTM(64),
#     keras.layers.Dense(32, activation="relu"),
#     keras.layers.Dense(1)
# ])

#Model 2 (GRU alternative)
model = keras.Sequential([
    keras.layers.Input(shape=(sequence_length, train_df.shape[1])),

    keras.layers.GRU(32),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dense(1)
])

In [30]:
model.compile(
    optimizer=keras.optimizers.Adam(0.001),
    loss="mse",
    metrics=["mae"]
)

In [31]:
callbacks = [
    keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks
)

Epoch 1/20
127/127 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - loss: 0.2502 - mae: 0.3941 - val_loss: 0.1605 - val_mae: 0.3146
Epoch 2/20
127/127 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.1796 - mae: 0.3336 - val_loss: 0.1606 - val_mae: 0.3044
Epoch 3/20
127/127 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.1673 - mae: 0.3200 - val_loss: 0.1526 - val_mae: 0.2945
Epoch 4/20
127/127 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.1371 - mae: 0.2902 - val_loss: 0.1266 - val_mae: 0.2717
Epoch 5/20
127/127 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - loss: 0.1211 - mae: 0.2711 - val_loss: 0.1145 - val_mae: 0.2568
Epoch 6/20
127/127 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - loss: 0.1130 - mae: 0.2604 - val_loss: 0.1059 - val_mae: 0.2453
Epoch 7/20
127/127 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - loss: 0.1052 - mae: 0.2510 - val_loss: 0.1009 - val_mae: 0.2376
Epoch 8/20
127/127 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.1005 - mae: 0.2453 - val_loss: 0.0981 - val_mae: 0.2327
Epoch 9/20
127/127 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/

#test best model

In [32]:
test_loss, test_mae = model.evaluate(test_ds)
print("Test MAE:", test_mae)

27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0752 - mae: 0.2194
Test MAE: 0.2194429486989975


#Predictions with matrices

In [33]:
y_true, y_pred = [], []

for x, y in test_ds:
    preds = model.predict(x)
    y_true.extend(y.numpy())
    y_pred.extend(preds.flatten())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

from sklearn.metrics import mean_absolute_error, mean_squared_error

print("MAE:", mean_absolute_error(y_true, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_true, y_pred)))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 233ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━